# 04 — Tableau Export
**Project:** OULAD Learning Analytics — Portfolio Project

This notebook exports clean, Tableau-ready CSV files for refreshing the Tableau Public workbook.

## Exports
| File | Tableau View |
|------|-------------|
| `tableau_funnel.csv` | Conversion Funnel Bar Chart |
| `tableau_cohort_retention.csv` | Cohort Retention Heatmap |
| `tableau_geo.csv` | Geographic Distribution (UK Regions) |
| `tableau_engagement_vs_result.csv` | Engagement vs Outcome Scatter |
| `tableau_weekly_activity.csv` | Weekly Activity Timeline |

In [1]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
np.random.seed(42)
import os

PROC    = '../data/processed/'
RAW     = '../data/raw/'
TABLEAU = PROC + 'tableau/'
os.makedirs(TABLEAU, exist_ok=True)

master      = pd.read_csv(PROC + 'master_students.csv')
student_vle = pd.read_csv(RAW + 'studentVle.csv')
vle         = pd.read_csv(RAW + 'vle.csv')  # activity type metadata

print(f'Master: {master.shape}')
print(f'Student VLE: {student_vle.shape}')

Master: (32593, 30)
Student VLE: (10655280, 6)


---
## Export 1: Funnel Data

In [2]:
total         = len(master)
n_active      = (master['total_clicks'] > 0).sum()
n_submitted   = (master['submission_count'] > 0).sum()
n_passed      = master['completed'].sum()
n_distinction = (master['final_result'] == 'Distinction').sum()

funnel_df = pd.DataFrame({
    'stage_order': [1, 2, 3, 4, 5],
    'stage_name':  ['Registered', 'Active (Any Click)', 'Scored Assessment', 'Passed', 'Distinction'],
    'students':    [total, n_active, n_submitted, n_passed, n_distinction]
})
funnel_df['pct_of_registered'] = (funnel_df['students'] / total * 100).round(1)
funnel_df['drop_off_pct'] = (100 - funnel_df['pct_of_registered']).round(1)

funnel_df.to_csv(TABLEAU + 'tableau_funnel.csv', index=False)
print('Saved tableau_funnel.csv')
funnel_df

Saved tableau_funnel.csv


,stage_order,stage_name,students,pct_of_registered,drop_off_pct
0,1,Registered,32593,100.0,0.0
1,2,Active (Any Click),29228,89.7,10.3
2,3,Scored Assessment,25820,79.2,20.8
3,4,Passed,15385,47.2,52.8
4,5,Distinction,3024,9.3,90.7


## Export 2 — cohort retention

Export the canonical weekly counts and peak-based index calculated by notebook 02. Check the archived published extract across the entire cohort/week grid.

In [3]:
sv = student_vle.copy()
sv['week'] = (sv['date'] // 7).astype(int)
sv = sv[sv['week'].between(0, 25)]

sv = sv.merge(
    master[['id_student','code_module','code_presentation']].drop_duplicates(),
    on=['id_student','code_module','code_presentation'], how='inner'
)

weekly_active = pd.read_csv(PROC + 'weekly_retention.csv').rename(columns={'code_presentation': 'cohort'})
weekly_active['retention_rate'] = weekly_active['retention_rate'].mul(100).round(1)
assert not weekly_active.duplicated(['cohort', 'week']).any()
weekly_active.to_csv(TABLEAU + 'tableau_cohort_retention.csv', index=False)
print("Saved peak-based retention percentages")

Saved peak-based retention percentages


In [4]:
legacy = pd.read_csv('../outputs/tableau/tableau_cohort_retention.csv')
comparison = legacy.merge(weekly_active, on=['cohort', 'week'], suffixes=('_published', '_refresh'),
                          how='outer', validate='one_to_one', indicator=True)
assert comparison['_merge'].eq('both').all()
cols = ['cohort', 'week', 'retention_rate_published', 'retention_rate_refresh']
print(comparison.loc[comparison.cohort.eq('2013B') & comparison.week.le(2), cols].to_string(index=False))
print(f"Checked {len(comparison)} cells; legacy >100%: {(comparison.retention_rate_published > 100).sum()}; "
      f"refresh >100%: {(comparison.retention_rate_refresh > 100).sum()}")
assert comparison.retention_rate_refresh.between(0, 100).all()

cohort  week  retention_rate_published  retention_rate_refresh
 2013B     0                     100.0                    96.7
 2013B     1                      96.8                    93.6
 2013B     2                     103.5                   100.0
Checked 104 cells; legacy >100%: 3; refresh >100%: 0


---
## Export 3: Geographic Distribution (UK Regions)

Regional mean score averages enrollment means among enrollments with at least one recorded mark.

In [5]:
# OULAD has 'region' field in studentInfo
geo_df = (
    master
    .groupby('region')
    .agg(
        total_students   = ('id_student', 'count'),
        completion_rate  = ('completed', 'mean'),
        avg_score        = ('avg_score', 'mean'),
        withdrawal_rate  = ('withdrew', 'mean'),
        avg_clicks       = ('total_clicks', 'mean')
    )
    .reset_index()
)

geo_df['completion_rate_pct'] = (geo_df['completion_rate'] * 100).round(1)
geo_df['withdrawal_rate_pct'] = (geo_df['withdrawal_rate'] * 100).round(1)
geo_df['avg_score']           = geo_df['avg_score'].round(1)
geo_df['avg_clicks']          = geo_df['avg_clicks'].round(0).astype(int)

geo_df.to_csv(TABLEAU + 'tableau_geo.csv', index=False)
print('Saved tableau_geo.csv')
geo_df.sort_values('total_students', ascending=False)

Saved tableau_geo.csv


,region,total_students,completion_rate,avg_score,withdrawal_rate,avg_clicks,completion_rate_pct,withdrawal_rate_pct
6,Scotland,3446,0.489553,74.4,0.263204,1387,49.0,26.3
0,East Anglian Region,3340,0.489820,73.1,0.301198,1184,49.0,30.1
3,London Region,3216,0.423507,71.1,0.345149,1160,42.4,34.5
8,South Region,3092,0.523933,73.7,0.300129,1316,52.4,30.0
5,North Western Region,2906,0.401927,71.1,0.355816,1123,40.2,35.6
11,West Midlands Region,2582,0.432998,72.2,0.353602,1084,43.3,35.4
9,South West Region,2436,0.497947,72.6,0.311166,1187,49.8,31.1
1,East Midlands Region,2365,0.457082,72.5,0.347569,1105,45.7,34.8
7,South East Region,2111,0.514922,74.4,0.307437,1234,51.5,30.7
10,Wales,2086,0.451582,71.8,0.251198,1066,45.2,25.1


---
## Export 4: Engagement vs Outcome

In [6]:
# Sample to avoid overplotting in Tableau (max 10k rows)
scatter_df = master[[
    'id_student', 'code_module', 'code_presentation',
    'total_clicks', 'active_days', 'avg_score',
    'submission_rate', 'engagement_tier',
    'final_result', 'completed', 'withdrew',
    'gender', 'age_band', 'highest_education', 'region'
]].copy()

# Cap clicks at 99th percentile to avoid extreme outliers distorting chart
p99 = scatter_df['total_clicks'].quantile(0.99)
scatter_df['total_clicks_capped'] = scatter_df['total_clicks'].clip(upper=p99)

if len(scatter_df) > 10000:
    scatter_df = scatter_df.sample(10000, random_state=42)

scatter_df.to_csv(TABLEAU + 'tableau_engagement_vs_result.csv', index=False)
print(f'Saved tableau_engagement_vs_result.csv ({len(scatter_df):,} rows)')

Saved tableau_engagement_vs_result.csv (10,000 rows)


---
## Export 5: Weekly Activity Timeline

In [7]:
sv_with_type = sv.merge(
    vle[['id_site', 'activity_type']],
    on='id_site', how='left'
)

weekly_clicks = (
    sv_with_type
    .groupby(['week', 'activity_type'])['sum_click']
    .sum()
    .reset_index(name='total_clicks')
)

weekly_clicks.to_csv(TABLEAU + 'tableau_weekly_activity.csv', index=False)
print('Saved tableau_weekly_activity.csv')
print(f'Activity types: {weekly_clicks["activity_type"].nunique()}')
weekly_clicks.head(10)

Saved tableau_weekly_activity.csv
Activity types: 20


,week,activity_type,total_clicks
0,0,dataplus,57
1,0,dualpane,427
2,0,externalquiz,4167
3,0,forumng,682793
4,0,glossary,6106
5,0,homepage,399927
6,0,htmlactivity,2705
7,0,oucollaborate,6793
8,0,oucontent,345619
9,0,ouelluminate,2718


In [8]:
# Verify all exports
print('=== Tableau Export Summary ===')
for f in sorted(os.listdir(TABLEAU)):
    if f.endswith('.csv'):
        df = pd.read_csv(TABLEAU + f)
        size = os.path.getsize(TABLEAU + f) / 1024
        print(f'  {f:<45} {df.shape[0]:>6,} rows × {df.shape[1]:>2} cols  ({size:.0f} KB)')

=== Tableau Export Summary ===
  tableau_cohort_retention.csv                     104 rows ×  5 cols  (2 KB)
  tableau_engagement_vs_result.csv              10,000 rows × 16 cols  (1045 KB)
  tableau_funnel.csv                                 5 rows ×  5 cols  (0 KB)
  tableau_geo.csv                                   13 rows ×  8 cols  (1 KB)
  tableau_weekly_activity.csv                      475 rows ×  3 cols  (8 KB)
